# 🛰️ SatQuery AI — Model 1: Remote Sensing VQA

**Team Spectra | Smart India Hackathon 2026**

This notebook fine-tunes **BLIP-2** (Vision-Language Model) with **QLoRA** for
Visual Question Answering on both **Optical** (Sentinel-2) and **SAR** remote
sensing imagery. The model generates free-form natural-language answers.

### Pipeline
1. Auto-collect ~200 remote sensing images (100 optical + 100 SAR)
2. Generate diverse natural-language QA pairs (~1 000 pairs)
3. Fine-tune BLIP-2 OPT-2.7B with 4-bit quantization + LoRA
4. Evaluate with BLEU, ROUGE-L metrics
5. Generate visualizations & save everything to Google Drive

**Requirements:** Google Colab with T4 GPU (free tier works)

---
## 1 · Environment Setup

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q "transformers>=4.36.0" "peft>=0.7.0" "bitsandbytes>=0.41.0" \
    "accelerate>=0.25.0" "datasets>=2.16.0" evaluate nltk rouge-score \
    pillow matplotlib seaborn tqdm scipy

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, json, random, csv, gc, warnings, time
from io import BytesIO
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import nltk
for _pkg in ("punkt", "wordnet", "punkt_tab"):
    nltk.download(_pkg, quiet=True)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "font.size": 11})
print("✅ All packages imported successfully")

In [ ]:
# ── Mount Google Drive & Configuration ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

CONFIG = dict(
    seed            = 42,
    drive_output    = "/content/drive/MyDrive/SatQuery_AI/Model1_VQA",
    data_dir        = "/content/rs_vqa_data",
    num_optical     = 100,
    num_sar         = 100,
    image_size      = 224,
    model_name      = "Salesforce/blip2-opt-2.7b",
    lora_rank       = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    learning_rate   = 2e-4,
    num_epochs      = 8,
    batch_size      = 2,
    grad_accum_steps= 4,
    max_length      = 256,
    warmup_ratio    = 0.1,
    val_split       = 0.2,
    early_stop_patience = 2,
)

# Create output directories
for _sub in ("model", "model/processor", "results", "dataset", "report"):
    os.makedirs(f"{CONFIG['drive_output']}/{_sub}", exist_ok=True)
for _sub in ("optical", "sar"):
    os.makedirs(f"{CONFIG['data_dir']}/{_sub}", exist_ok=True)

# Reproducibility
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Device : {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

with open(f"{CONFIG['drive_output']}/config.json", "w") as _f:
    json.dump(CONFIG, _f, indent=2)
print(f"\n📁 Output : {CONFIG['drive_output']}")
print("✅ Setup complete")

---
## 2 · Dataset Collection

| Source | Modality | Details |
|--------|----------|---------|
| **EuroSAT** (HuggingFace) | Optical | Sentinel-2 RGB, 10 land-use classes |
| **BigEarthNet-S1** or **Synthetic SAR** | SAR | Sentinel-1 patches / simulated speckle |

The notebook tries real SAR data first and falls back to SAR synthesis for
the prototype.

In [ ]:
# ── 2a  Download Optical Images (EuroSAT) ───────────────────────────────────
from datasets import load_dataset

print("=" * 60)
print("📡  STEP 1 : Downloading EuroSAT (Optical)")
print("=" * 60)

EUROSAT_CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake",
]

eurosat_ds = None
for _src in ("blanchon/EuroSAT", "tanganke/EuroSAT"):
    try:
        print(f"  Trying {_src} …")
        eurosat_ds = load_dataset(_src, split="train", trust_remote_code=True)
        print(f"  ✅ Loaded: {len(eurosat_ds)} images")
        break
    except Exception as _e:
        print(f"  ❌ {_e}")

# Fallback: download ZIP from Zenodo
if eurosat_ds is None:
    print("\n  Downloading from Zenodo …")
    os.system(
        "wget -q https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip "
        "-O /content/eurosat.zip && "
        "unzip -q -o /content/eurosat.zip -d /content/eurosat_raw/"
    )
    _root = Path("/content/eurosat_raw")
    # Find the actual image directory
    _candidates = list(_root.rglob("*.jpg")) + list(_root.rglob("*.tif"))
    print(f"  Found {len(_candidates)} image files")

optical_samples = []
per_class = CONFIG["num_optical"] // len(EUROSAT_CLASSES)

if eurosat_ds is not None:
    # ── HuggingFace path ──────────────────────────────────
    class_counts = defaultdict(int)
    indices = list(range(len(eurosat_ds)))
    random.shuffle(indices)

    for idx in indices:
        sample = eurosat_ds[idx]
        label  = sample["label"]
        cls    = EUROSAT_CLASSES[label] if label < len(EUROSAT_CLASSES) else f"class_{label}"
        if class_counts[cls] >= per_class:
            continue
        img = sample["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert("RGB")
        _path = f"{CONFIG['data_dir']}/optical/{cls}_{class_counts[cls]:02d}.png"
        img.save(_path)
        optical_samples.append(dict(image_path=_path, cls=cls, modality="optical"))
        class_counts[cls] += 1
        if len(optical_samples) >= CONFIG["num_optical"]:
            break
else:
    # ── Zenodo fallback ───────────────────────────────────
    _root = Path("/content/eurosat_raw")
    for cls_dir in sorted(p for p in _root.rglob("*") if p.is_dir()):
        cls = cls_dir.name
        imgs = sorted(cls_dir.glob("*.*"))
        for i, ip in enumerate(imgs[:per_class]):
            _path = f"{CONFIG['data_dir']}/optical/{cls}_{i:02d}.png"
            Image.open(ip).convert("RGB").save(_path)
            optical_samples.append(dict(image_path=_path, cls=cls, modality="optical"))

del eurosat_ds; gc.collect()
print(f"\n  Collected {len(optical_samples)} optical images")

In [ ]:
# ── 2b  Acquire SAR Images ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📡  STEP 2 : Acquiring SAR Images")
print("=" * 60)

sar_samples  = []
sar_source   = "unknown"

# ── Attempt 1: BigEarthNet-S1 streaming from HuggingFace ─────────────────
try:
    print("  Attempting BigEarthNet-S1 (HuggingFace streaming) …")
    _ben = load_dataset(
        "BIFOLD-BigEarthNet/BigEarthNet-S1",
        split="train", streaming=True, trust_remote_code=True,
    )
    _count = 0
    for _s in tqdm(_ben, total=CONFIG["num_sar"], desc="  Downloading SAR"):
        if _count >= CONFIG["num_sar"]:
            break
        _img = None
        for _k in ("image", "s1", "vv", "vh"):
            if _k in _s:
                _img = _s[_k]; break
        if _img is None:
            continue
        if isinstance(_img, Image.Image):
            _img = _img.convert("L")
        else:
            _arr = np.array(_img)
            if _arr.ndim == 3:
                _arr = _arr.mean(axis=-1)
            _img = Image.fromarray(_arr.astype(np.uint8), mode="L")
        _path = f"{CONFIG['data_dir']}/sar/sar_{_count:04d}.png"
        _img.save(_path)
        _labels = _s.get("labels", _s.get("label", ["unknown"]))
        if isinstance(_labels, (int, np.integer)):
            _labels = [f"class_{_labels}"]
        elif isinstance(_labels, str):
            _labels = [_labels]
        sar_samples.append(dict(
            image_path=_path,
            cls=_labels[0] if _labels else "unknown",
            modality="sar",
        ))
        _count += 1
    if len(sar_samples) < CONFIG["num_sar"] // 2:
        raise ValueError(f"Only got {len(sar_samples)} patches – not enough")
    sar_source = "BigEarthNet-S1"
    print(f"  ✅ Loaded {len(sar_samples)} SAR images")
except Exception as _e:
    print(f"  ❌ BigEarthNet-S1 failed: {_e}")
    print("\n  🔄 Fallback : Generating SAR-like images from optical data")
    print("     (grayscale → multiplicative speckle noise → log transform)")
    print("     ⚠️  Replace with real Sentinel-1 data for production\n")
    sar_samples = []
    for i, _s in enumerate(tqdm(optical_samples, desc="  Generating SAR")):
        _arr = np.array(Image.open(_s["image_path"]).convert("L"), dtype=np.float64)
        _speckle = np.random.gamma(shape=4, scale=0.25, size=_arr.shape)
        _arr = _arr * _speckle
        _arr = np.clip(_arr, 1, None)
        _arr = 10.0 * np.log10(_arr)
        _arr = (_arr - _arr.min()) / (_arr.max() - _arr.min() + 1e-8) * 255
        _path = f"{CONFIG['data_dir']}/sar/sar_{i:04d}.png"
        Image.fromarray(_arr.astype(np.uint8), mode="L").save(_path)
        sar_samples.append(dict(image_path=_path, cls=_s["cls"], modality="sar"))
    sar_source = "Synthetic (EuroSAT + Speckle)"
    print(f"  ✅ Generated {len(sar_samples)} synthetic SAR images")

gc.collect()
print(f"\n📊 Total : {len(optical_samples)} optical + {len(sar_samples)} SAR"
      f" = {len(optical_samples)+len(sar_samples)} images  ({sar_source})")

In [ ]:
# ── 2c  Generate Natural-Language QA Pairs ───────────────────────────────────
print("\n" + "=" * 60)
print("💬  STEP 3 : Generating Natural-Language QA Pairs")
print("=" * 60)

CLASS_INFO = {
    "AnnualCrop": dict(
        desc="agricultural fields with annual crops showing seasonal planting patterns",
        feat=["crop rows", "planting patterns", "agricultural fields", "seasonal vegetation"],
        cat="agricultural", water=False, urban=False),
    "Forest": dict(
        desc="dense forest cover with continuous tree canopy and woodland areas",
        feat=["tree canopy", "dense vegetation", "forest cover", "woodland areas"],
        cat="vegetation", water=False, urban=False),
    "HerbaceousVegetation": dict(
        desc="grasslands and herbaceous vegetation with open green areas",
        feat=["grassland", "natural vegetation", "herbaceous cover", "open meadows"],
        cat="vegetation", water=False, urban=False),
    "Highway": dict(
        desc="major road infrastructure with highway corridors and transportation networks",
        feat=["highway lanes", "road infrastructure", "transportation corridors", "paved surfaces"],
        cat="infrastructure", water=False, urban=True),
    "Industrial": dict(
        desc="industrial zones with large buildings, warehouses, and manufacturing facilities",
        feat=["industrial buildings", "warehouses", "large structures", "factory complexes"],
        cat="urban", water=False, urban=True),
    "Pasture": dict(
        desc="pastoral grasslands used for livestock grazing with open meadows",
        feat=["grazing land", "open meadows", "pastoral areas", "livestock fields"],
        cat="agricultural", water=False, urban=False),
    "PermanentCrop": dict(
        desc="permanent crop plantations such as orchards, vineyards, or fruit trees",
        feat=["orchard trees", "permanent plantations", "vineyard rows", "perennial crops"],
        cat="agricultural", water=False, urban=False),
    "Residential": dict(
        desc="residential areas with housing developments, roads, and neighbourhood patterns",
        feat=["houses", "residential buildings", "urban roads", "neighbourhood layout"],
        cat="urban", water=False, urban=True),
    "River": dict(
        desc="river channels and riparian zones with flowing water and bank vegetation",
        feat=["water channel", "river banks", "riparian vegetation", "flowing water"],
        cat="water", water=True, urban=False),
    "SeaLake": dict(
        desc="large water bodies including seas, lakes, coastal areas, and reservoirs",
        feat=["open water", "lake surface", "coastal areas", "water body"],
        cat="water", water=True, urban=False),
}

def _info(cls):
    if cls in CLASS_INFO:
        return CLASS_INFO[cls]
    return dict(desc=cls.lower().replace("_", " "), feat=[cls.lower()],
                cat="general", water=False, urban=False)

def generate_qa(sample):
    """Return 5 diverse QA dicts for one image sample."""
    c   = sample["cls"]
    mod = sample["modality"]
    inf = _info(c)
    mn  = "satellite" if mod == "optical" else "SAR"
    qa  = []

    # Q1 – scene description
    qa.append(dict(
        question=f"Describe the main features visible in this {mn} image.",
        answer=(f"This {mn} image shows {inf['desc']}. "
                f"Key features include {', '.join(inf['feat'][:3])}.")))

    # Q2 – land-cover ID
    qa.append(dict(
        question="What type of land cover is shown in this image?",
        answer=(f"The image primarily displays {inf['desc']}. "
                f"This falls under the {inf['cat']} land-cover category.")))

    # Q3 – water check
    if inf["water"]:
        qa.append(dict(
            question="Is there any water body visible in this image?",
            answer=(f"Yes, the image clearly shows water features. "
                    f"It contains {inf['desc']}, with {inf['feat'][0]} being prominent.")))
    else:
        qa.append(dict(
            question="Is there any water body visible in this image?",
            answer=(f"No significant water body is visible. "
                    f"The scene is dominated by {inf['desc']}.")))

    # Q4 – dominant feature
    qa.append(dict(
        question="What is the dominant feature in this scene?",
        answer=(f"The dominant feature is {inf['feat'][0]}, "
                f"characteristic of {inf['desc']}.")))

    # Q5 – modality-specific
    if mod == "sar":
        qa.append(dict(
            question="What information can you extract from this SAR image?",
            answer=(f"This SAR image reveals {inf['desc']} through radar "
                    f"backscatter patterns. The radar signature indicates "
                    f"{inf['feat'][0]} with texture typical of {inf['cat']} areas.")))
    else:
        qa.append(dict(
            question="Describe the spectral characteristics in this optical image.",
            answer=(f"The optical image shows spectral signatures consistent with "
                    f"{inf['desc']}. Reflectance patterns indicate {inf['feat'][0]} "
                    f"with surrounding {inf['feat'][-1]}.")))
    return qa

# Build full QA dataset
all_samples = optical_samples + sar_samples
all_qa = []
for s in tqdm(all_samples, desc="Generating QA"):
    for qa in generate_qa(s):
        all_qa.append(dict(
            image_path=s["image_path"], modality=s["modality"],
            cls=s["cls"], question=qa["question"], answer=qa["answer"]))

random.shuffle(all_qa)
_split = int(len(all_qa) * (1 - CONFIG["val_split"]))
train_data = all_qa[:_split]
val_data   = all_qa[_split:]

print(f"\n📊 QA Dataset")
print(f"   Total    : {len(all_qa)}")
print(f"   Train    : {len(train_data)}")
print(f"   Val      : {len(val_data)}")
print(f"   Optical  : {sum(1 for x in all_qa if x['modality']=='optical')}")
print(f"   SAR      : {sum(1 for x in all_qa if x['modality']=='sar')}")

for _name, _data in [("train_qa_pairs.json", train_data), ("val_qa_pairs.json", val_data)]:
    with open(f"{CONFIG['drive_output']}/dataset/{_name}", "w") as f:
        json.dump(_data, f, indent=2)
print("💾 Saved to Drive")

In [ ]:
# ── 2d  Dataset Distribution Visualisation ───────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
_cd = defaultdict(int)
for s in all_qa:
    _cd[s["cls"]] += 1
_cls = sorted(_cd); _cnt = [_cd[c] for c in _cls]
axes[0].barh(_cls, _cnt, color=sns.color_palette("husl", len(_cls)))
axes[0].set_xlabel("QA Pairs"); axes[0].set_title("Per-Class QA Count", fontweight="bold")

# Modality pie
_md = defaultdict(int)
for s in all_qa:
    _md[s["modality"]] += 1
axes[1].pie(_md.values(),
    labels=[f"{k.upper()}\n({v})" for k, v in _md.items()],
    autopct="%1.0f%%", colors=["#3498db", "#e74c3c"],
    startangle=90, explode=(0.05, 0.05))
axes[1].set_title("Modality Split", fontweight="bold")

# Train / val
axes[2].bar(["Train", "Val"], [len(train_data), len(val_data)],
            color=["#2ecc71", "#f39c12"])
axes[2].set_ylabel("QA Pairs"); axes[2].set_title("Train / Val Split", fontweight="bold")
for i, v in enumerate([len(train_data), len(val_data)]):
    axes[2].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.suptitle("Dataset Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/dataset_distribution.png", bbox_inches="tight")
plt.show()

# Sample images
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Sample Images — Optical (top) · SAR (bottom)", fontsize=14, fontweight="bold")
for i in range(5):
    axes[0, i].imshow(Image.open(optical_samples[i * 2]["image_path"]))
    axes[0, i].set_title(optical_samples[i * 2]["cls"], fontsize=9); axes[0, i].axis("off")
    axes[1, i].imshow(Image.open(sar_samples[i * 2]["image_path"]).convert("L"), cmap="gray")
    axes[1, i].set_title(sar_samples[i * 2]["cls"], fontsize=9); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Optical", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("SAR", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/sample_images.png", bbox_inches="tight")
plt.show()
print("✅ Distribution & sample plots saved")

---
## 3 · Data Preprocessing & DataLoaders

In [ ]:
from transformers import Blip2Processor

print("=" * 60)
print("⚙️  STEP 4 : Data Pipeline")
print("=" * 60)

processor = Blip2Processor.from_pretrained(CONFIG["model_name"])

class RSVQADataset(Dataset):
    """Remote-Sensing VQA dataset – returns image + question → answer."""

    def __init__(self, data, processor, max_length=256):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        q, a = item["question"], item["answer"]

        # Full prompt for causal-LM training
        prompt = f"Question: {q} Answer: {a}"
        enc = self.processor(
            images=image, text=prompt, return_tensors="pt",
            padding="max_length", max_length=self.max_length, truncation=True,
        )

        pv  = enc["pixel_values"].squeeze(0)
        ids = enc["input_ids"].squeeze(0)
        am  = enc["attention_mask"].squeeze(0)

        # Labels: mask prompt tokens, keep only answer for loss
        labels = ids.clone()
        _prompt_only = f"Question: {q} Answer:"
        _plen = len(self.processor.tokenizer(_prompt_only, add_special_tokens=True)["input_ids"])
        labels[:_plen] = -100
        labels[am == 0] = -100

        return dict(pixel_values=pv, input_ids=ids, attention_mask=am,
                    labels=labels, question=q, answer=a,
                    modality=item["modality"], cls=item["cls"])


def collate_fn(batch):
    return dict(
        pixel_values  = torch.stack([b["pixel_values"]  for b in batch]),
        input_ids     = torch.stack([b["input_ids"]     for b in batch]),
        attention_mask= torch.stack([b["attention_mask"] for b in batch]),
        labels        = torch.stack([b["labels"]        for b in batch]),
        questions     = [b["question"]  for b in batch],
        answers       = [b["answer"]    for b in batch],
        modalities    = [b["modality"]  for b in batch],
        classes       = [b["cls"]       for b in batch],
    )

train_ds = RSVQADataset(train_data, processor, CONFIG["max_length"])
val_ds   = RSVQADataset(val_data,   processor, CONFIG["max_length"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          shuffle=True,  collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                          shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

_sb = next(iter(train_loader))
print(f"\n📊 Loaders ready")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val   batches : {len(val_loader)}")
print(f"   pixel_values  : {_sb['pixel_values'].shape}")
print(f"   input_ids     : {_sb['input_ids'].shape}")
print("✅ Data pipeline ready")

---
## 4 · Model Setup — BLIP-2 + QLoRA

In [ ]:
from transformers import Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("=" * 60)
print("🧠  STEP 5 : Loading BLIP-2 with QLoRA")
print("=" * 60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("  Loading model (2-3 min) …")
model = Blip2ForConditionalGeneration.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

# Freeze vision encoder + Q-Former
for p in model.vision_model.parameters():
    p.requires_grad = False
for p in model.qformer.parameters():
    p.requires_grad = False
print("  ✅ Vision encoder & Q-Former frozen")

lora_cfg = LoraConfig(
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)

_train_p, _total_p = model.get_nb_trainable_parameters()
print(f"\n📊 Parameters")
print(f"   Total      : {_total_p:>12,}")
print(f"   Trainable  : {_train_p:>12,}  ({100*_train_p/_total_p:.2f}%)")
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
print("\n✅ Model ready")

---
## 5 · Training Loop

In [ ]:
from transformers import get_cosine_schedule_with_warmup

print("=" * 60)
print("🏋️  STEP 6 : Training")
print("=" * 60)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"],
                              weight_decay=0.01, betas=(0.9, 0.999))

_total_steps  = len(train_loader) * CONFIG["num_epochs"] // CONFIG["grad_accum_steps"]
_warmup_steps = int(_total_steps * CONFIG["warmup_ratio"])
scheduler = get_cosine_schedule_with_warmup(optimizer, _warmup_steps, _total_steps)

scaler = torch.cuda.amp.GradScaler()
training_log = []
best_val_loss = float("inf")
patience_ctr  = 0
best_epoch    = 0
all_lrs       = []

print(f"   Epochs         : {CONFIG['num_epochs']}")
print(f"   Batch size     : {CONFIG['batch_size']}  (eff. {CONFIG['batch_size']*CONFIG['grad_accum_steps']})")
print(f"   Optim. steps   : {_total_steps}")
print(f"   Warmup steps   : {_warmup_steps}")
print(f"   Learning rate  : {CONFIG['learning_rate']}\n")

t0 = time.time()

for epoch in range(CONFIG["num_epochs"]):
    # ── Train ────────────────────────────────────────────────────
    model.train()
    _tloss, _tn = 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Train]")
    for step, batch in enumerate(pbar):
        with torch.cuda.amp.autocast():
            out = model(
                pixel_values=batch["pixel_values"].to(device),
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            loss = out.loss / CONFIG["grad_accum_steps"]
        scaler.scale(loss).backward()

        if (step + 1) % CONFIG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            all_lrs.append(scheduler.get_last_lr()[0])

        _tloss += loss.item() * CONFIG["grad_accum_steps"]
        _tn += 1
        pbar.set_postfix(loss=f"{_tloss/_tn:.4f}")

    avg_train = _tloss / _tn

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    _vloss, _vn = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Val]"):
            with torch.cuda.amp.autocast():
                out = model(
                    pixel_values=batch["pixel_values"].to(device),
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device),
                )
            _vloss += out.loss.item(); _vn += 1
    avg_val = _vloss / _vn

    _lr = scheduler.get_last_lr()[0]
    training_log.append(dict(epoch=epoch+1, train_loss=round(avg_train, 4),
                             val_loss=round(avg_val, 4), lr=_lr,
                             timestamp=datetime.now().isoformat()))

    print(f"\n📊 Epoch {epoch+1}: train={avg_train:.4f}  val={avg_val:.4f}  lr={_lr:.2e}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val; best_epoch = epoch + 1; patience_ctr = 0
        model.save_pretrained(f"{CONFIG['drive_output']}/model")
        processor.save_pretrained(f"{CONFIG['drive_output']}/model/processor")
        print(f"   💾 Best model saved (val={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        print(f"   ⚠️  No improvement ({patience_ctr}/{CONFIG['early_stop_patience']})")

    if patience_ctr >= CONFIG["early_stop_patience"]:
        print(f"\n🛑 Early stopping at epoch {epoch+1}"); break

elapsed = time.time() - t0
print(f"\n✅ Training done in {elapsed/60:.1f} min  |  Best epoch {best_epoch}  val={best_val_loss:.4f}")

# Save logs
with open(f"{CONFIG['drive_output']}/results/training_log.json", "w") as f:
    json.dump(training_log, f, indent=2)
with open(f"{CONFIG['drive_output']}/results/training_log.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_loss", "lr", "timestamp"])
    w.writeheader(); w.writerows(training_log)

---
## 6 · Evaluation — Metrics & Generation

In [ ]:
print("=" * 60)
print("📈  STEP 7 : Evaluation")
print("=" * 60)

# Reload best checkpoint
try:
    model.load_adapter(f"{CONFIG['drive_output']}/model", adapter_name="default")
    print("  ✅ Best checkpoint loaded")
except Exception:
    print("  ⚠️  Using current weights")

model.eval()

predictions  = []
references   = []
eval_details = []

with torch.no_grad():
    for item in tqdm(val_data, desc="Generating"):
        image  = Image.open(item["image_path"]).convert("RGB")
        prompt = f"Question: {item['question']} Answer:"
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

        gen_ids = model.generate(
            **inputs, max_new_tokens=128,
            do_sample=False, num_beams=3,
            repetition_penalty=1.2, length_penalty=1.0,
        )
        pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
        # Clean prompt artefacts
        if "Answer:" in pred:
            pred = pred.split("Answer:")[-1].strip()

        predictions.append(pred)
        references.append(item["answer"])
        eval_details.append(dict(
            question=item["question"], ground_truth=item["answer"],
            prediction=pred, modality=item["modality"],
            cls=item["cls"], image_path=item["image_path"]))

print(f"  Generated {len(predictions)} predictions")

# ── Compute metrics ──────────────────────────────────────────────────────────
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smooth = SmoothingFunction().method1
_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

bleu1, bleu4, rougeL = [], [], []
for p, r in zip(predictions, references):
    rt = r.lower().split(); pt = p.lower().split()
    bleu1.append(sentence_bleu([rt], pt, weights=(1, 0, 0, 0), smoothing_function=smooth))
    bleu4.append(sentence_bleu([rt], pt, weights=(.25, .25, .25, .25), smoothing_function=smooth))
    rougeL.append(_scorer.score(r, p)["rougeL"].fmeasure)

metrics = {
    "BLEU-1":  round(np.mean(bleu1), 4),
    "BLEU-4":  round(np.mean(bleu4), 4),
    "ROUGE-L": round(np.mean(rougeL), 4),
    "num_samples": len(predictions),
    "best_epoch":  best_epoch,
    "best_val_loss": round(best_val_loss, 4),
    "training_time_min": round(elapsed / 60, 1),
}
for _mod in ("optical", "sar"):
    _idx = [i for i, d in enumerate(eval_details) if d["modality"] == _mod]
    if _idx:
        metrics[f"BLEU-1_{_mod}"]  = round(np.mean([bleu1[i] for i in _idx]), 4)
        metrics[f"BLEU-4_{_mod}"]  = round(np.mean([bleu4[i] for i in _idx]), 4)
        metrics[f"ROUGE-L_{_mod}"] = round(np.mean([rougeL[i] for i in _idx]), 4)

print("\n" + "=" * 40)
print("📊  EVALUATION RESULTS")
print("=" * 40)
for k, v in metrics.items():
    print(f"  {k:>20s} : {v}")

with open(f"{CONFIG['drive_output']}/results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(f"{CONFIG['drive_output']}/results/eval_details.json", "w") as f:
    json.dump(eval_details, f, indent=2)
print("\n✅ Metrics saved")

---
## 7 · Visualisations & Result Plots

In [ ]:
# ── 7a  Loss Curves & LR Schedule ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
_ep = [x["epoch"] for x in training_log]
axes[0].plot(_ep, [x["train_loss"] for x in training_log], "o-", label="Train", color="#3498db", lw=2)
axes[0].plot(_ep, [x["val_loss"]   for x in training_log], "s-", label="Val",   color="#e74c3c", lw=2)
axes[0].axvline(best_epoch, ls="--", color="#2ecc71", alpha=.7, label=f"Best (ep {best_epoch})")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss", fontweight="bold"); axes[0].legend(); axes[0].grid(alpha=.3)

if all_lrs:
    axes[1].plot(all_lrs, color="#9b59b6", lw=1.5)
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("LR")
    axes[1].set_title("Learning Rate Schedule", fontweight="bold"); axes[1].grid(alpha=.3)
    axes[1].ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))

plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/loss_curve.png", bbox_inches="tight")
plt.show(); print("✅ Loss curves saved")

In [ ]:
# ── 7b  Metric Bar Charts ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
_mn = ["BLEU-1", "BLEU-4", "ROUGE-L"]
_mv = [metrics[k] for k in _mn]
bars = axes[0].bar(_mn, _mv, color=["#3498db", "#2ecc71", "#e74c3c"])
axes[0].set_ylim(0, max(_mv) * 1.4 + 0.05)
axes[0].set_title("Overall Metrics", fontweight="bold"); axes[0].set_ylabel("Score")
for b, v in zip(bars, _mv):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                 f"{v:.3f}", ha="center", fontweight="bold")

x = np.arange(len(_mn)); w = 0.35
_ov = [metrics.get(f"{m}_optical", 0) for m in _mn]
_sv = [metrics.get(f"{m}_sar", 0)     for m in _mn]
axes[1].bar(x - w/2, _ov, w, label="Optical", color="#3498db")
axes[1].bar(x + w/2, _sv, w, label="SAR",     color="#e74c3c")
axes[1].set_xticks(x); axes[1].set_xticklabels(_mn)
axes[1].set_title("Metrics by Modality", fontweight="bold")
axes[1].set_ylabel("Score"); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/metric_charts.png", bbox_inches="tight")
plt.show(); print("✅ Metric charts saved")

In [ ]:
# ── 7c  Sample Predictions Grid ─────────────────────────────────────────────
_opt_ev = [d for d in eval_details if d["modality"] == "optical"][:4]
_sar_ev = [d for d in eval_details if d["modality"] == "sar"][:4]
_samples = _opt_ev + _sar_ev

fig, axes = plt.subplots(2, 4, figsize=(24, 11))
for i, det in enumerate(_samples):
    r, c = i // 4, i % 4
    img = Image.open(det["image_path"]).convert("RGB")
    axes[r, c].imshow(img); axes[r, c].axis("off")
    _icon = "🛰️ OPT" if det["modality"] == "optical" else "📡 SAR"
    axes[r, c].set_title(
        f"{_icon}\nQ: {det['question'][:55]}…\n"
        f"GT: {det['ground_truth'][:60]}…\n"
        f"Pred: {det['prediction'][:60]}…",
        fontsize=7, ha="left", x=0, va="top")

plt.suptitle("Sample Predictions – Ground Truth vs Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/sample_predictions.png", bbox_inches="tight")
plt.show(); print("✅ Sample predictions saved")

In [ ]:
# ── 7d  Score Distribution & Best / Worst ────────────────────────────────────
_scored = sorted([{**d, "rl": rougeL[i]} for i, d in enumerate(eval_details)],
                 key=lambda x: x["rl"], reverse=True)

print("🏆 TOP-5 (highest ROUGE-L)")
print("-" * 80)
for s in _scored[:5]:
    print(f"  [{s['modality'].upper()}] RL={s['rl']:.3f}  Q: {s['question'][:70]}")
    print(f"    GT  : {s['ground_truth'][:70]}")
    print(f"    Pred: {s['prediction'][:70]}\n")

print("⚠️ BOTTOM-5 (lowest ROUGE-L)")
print("-" * 80)
for s in _scored[-5:]:
    print(f"  [{s['modality'].upper()}] RL={s['rl']:.3f}  Q: {s['question'][:70]}")
    print(f"    GT  : {s['ground_truth'][:70]}")
    print(f"    Pred: {s['prediction'][:70]}\n")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rougeL, bins=20, color="#3498db", edgecolor="white", alpha=.85)
ax.axvline(np.mean(rougeL), color="#e74c3c", ls="--", label=f"Mean={np.mean(rougeL):.3f}")
ax.set_xlabel("ROUGE-L"); ax.set_ylabel("Count")
ax.set_title("ROUGE-L Score Distribution", fontweight="bold"); ax.legend()
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/confusion_analysis.png", bbox_inches="tight")
plt.show(); print("✅ Score distribution saved")

---
## 8 · Export & Training Report

In [ ]:
report = f"""# SatQuery AI — Model 1 : RS-VQA Training Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Team:** Spectra | Smart India Hackathon 2026

---

## Configuration
| Key | Value |
|-----|-------|
| Model | `{CONFIG['model_name']}` |
| Fine-tuning | QLoRA  rank={CONFIG['lora_rank']}  alpha={CONFIG['lora_alpha']} |
| Train samples | {len(train_data)} |
| Val samples | {len(val_data)} |
| Optical images | {len(optical_samples)} |
| SAR images | {len(sar_samples)} ({sar_source}) |
| QA/image | 5 |
| Epochs trained | {len(training_log)} / {CONFIG['num_epochs']} |
| Best epoch | {best_epoch} |
| Training time | {elapsed/60:.1f} min |

## Results
| Metric | Overall | Optical | SAR |
|--------|---------|---------|-----|
| BLEU-1 | {metrics.get('BLEU-1','—')} | {metrics.get('BLEU-1_optical','—')} | {metrics.get('BLEU-1_sar','—')} |
| BLEU-4 | {metrics.get('BLEU-4','—')} | {metrics.get('BLEU-4_optical','—')} | {metrics.get('BLEU-4_sar','—')} |
| ROUGE-L | {metrics.get('ROUGE-L','—')} | {metrics.get('ROUGE-L_optical','—')} | {metrics.get('ROUGE-L_sar','—')} |

## Per-Epoch Loss
| Epoch | Train | Val |
|-------|-------|-----|
"""
for _l in training_log:
    report += f"| {_l['epoch']} | {_l['train_loss']} | {_l['val_loss']} |\n"

report += f"""
## Saved Artefacts
```
{CONFIG['drive_output']}/
├── model/           ← LoRA adapter + processor
├── results/         ← metrics, plots, logs
├── dataset/         ← train/val QA JSONs
└── report/          ← this report
```

## Notes
- Prototype trained on ~200 images for SIH demonstration.
- For production: use BigEarthNet-S1 or Sentinel-1 for real SAR.
- Model generates **natural-language** answers (not classification labels).
"""

with open(f"{CONFIG['drive_output']}/report/training_report.md", "w") as f:
    f.write(report)

print("✅ Training report saved")
print("\n" + "=" * 60)
print("🎉  PIPELINE COMPLETE")
print("=" * 60)
print(f"\n📁 All outputs → {CONFIG['drive_output']}")
print(f"\n   BLEU-1  = {metrics['BLEU-1']}")
print(f"   BLEU-4  = {metrics['BLEU-4']}")
print(f"   ROUGE-L = {metrics['ROUGE-L']}")
print(f"   Best epoch {best_epoch}   val_loss={best_val_loss:.4f}")

---
## 9 · Interactive Demo

Test the fine-tuned model with any image + question.

In [ ]:
def satquery_vqa(image_path, question):
    """Run VQA inference on a single remote-sensing image."""
    image  = Image.open(image_path).convert("RGB")
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=128,
                             do_sample=False, num_beams=3, repetition_penalty=1.2)
    ans = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    if "Answer:" in ans:
        ans = ans.split("Answer:")[-1].strip()
    return ans

# ── Demo queries ─────────────────────────────────────────────────────────────
print("🛰️  SatQuery AI — Interactive VQA Demo")
print("=" * 50)

_demos = [
    (optical_samples[0]["image_path"],  "Describe the main features visible in this image.", "optical"),
    (sar_samples[0]["image_path"],      "What information can you extract from this SAR image?", "sar"),
    (optical_samples[5]["image_path"],  "Is there any water body visible in this image?", "optical"),
    (sar_samples[5]["image_path"],      "What type of land cover is shown in this image?", "sar"),
]

fig, axes = plt.subplots(1, len(_demos), figsize=(5 * len(_demos), 5))
for i, (ip, q, mod) in enumerate(_demos):
    ans = satquery_vqa(ip, q)
    axes[i].imshow(Image.open(ip).convert("RGB")); axes[i].axis("off")
    _ic = "🛰️" if mod == "optical" else "📡"
    axes[i].set_title(f"{_ic} {mod.upper()}\nQ: {q[:50]}…\nA: {ans[:60]}…", fontsize=8)
    print(f"\n{_ic} [{mod.upper()}]")
    print(f"   Q: {q}")
    print(f"   A: {ans}")

plt.suptitle("SatQuery AI — VQA Demo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/demo_outputs.png", bbox_inches="tight")
plt.show()

print("\n✅ Demo complete — model ready for SatQuery AI integration!")